# MA3632 — Workshop 9: Ensemble Methods

This workshop accompanies Lecture 9. We verify the variance reduction argument
for bagging numerically, implement bagging from scratch, train and tune Random
Forests, compare MDI and permutation importance, and fit gradient boosting with
early stopping. The final part compares all three ensemble families on the same
dataset.

Work through all parts in order. Take-home exercises are at the end.

---

## Part A — The variance formula and bagging from scratch

The lecture showed that the variance of the average of $B$ estimators each with
variance $\sigma^2$ and pairwise correlation $\rho$ is

$$
\text{Var}\!\left(\bar{f}\right) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2.
$$

As $B \to \infty$ this tends to $\rho\,\sigma^2$, not zero.  We verify this
numerically before building a bagging classifier from scratch.

### A1. Imports and data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

# Digits dataset — 1797 samples, 64 features (8x8 pixel intensities), 10 classes
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig, y_dig, test_size=0.25, random_state=0, stratify=y_dig
)
print(f"Training set: {X_tr.shape}  Test set: {X_te.shape}")
print(f"Classes: {np.unique(y_dig)}")

### A2. Verifying the variance formula numerically

In [ ]:
def simulate_ensemble_variance(B, rho, sigma2=0.25, n_sim=8000, seed=0):
    """
    Simulate the variance of the mean of B correlated estimators.
    Each estimator has marginal variance sigma2; pairwise correlation rho
    is induced by a shared latent factor Z:
        f_b = sqrt(rho)*Z + sqrt(1-rho)*eps_b,  eps_b iid N(0, sigma2).
    """
    rng = np.random.default_rng(seed)
    Z   = rng.normal(0, np.sqrt(sigma2), n_sim)
    eps = rng.normal(0, np.sqrt(sigma2 * (1 - rho)), (n_sim, B))
    preds = np.sqrt(rho) * Z[:, None] + np.sqrt(1 - rho) * eps
    ensemble_mean = preds.mean(axis=1)
    return ensemble_mean.var()

B_values = [1, 5, 10, 25, 50, 100, 200]
rho_values = [0.0, 0.3, 0.6, 0.9]
sigma2 = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
for rho in rho_values:
    empirical = [simulate_ensemble_variance(B, rho, sigma2) for B in B_values]
    theoretical_floor = rho * sigma2
    ax.plot(B_values, empirical, marker='o', label=f'rho={rho}')
    ax.axhline(theoretical_floor, color='grey', lw=0.8, ls='--')

ax.set_xlabel('Number of trees B')
ax.set_ylabel('Variance of ensemble mean')
ax.set_title('Ensemble variance vs B for different pairwise correlations')
ax.legend()
plt.tight_layout()
plt.show()
print()
print("Theoretical floors (rho * sigma2):")
for rho in rho_values:
    print(f"  rho={rho}: floor = {rho * sigma2:.4f}")

**Exercise A.** Looking at the plot, explain in one or two sentences why increasing
$B$ beyond about 50 gives diminishing returns when $\rho$ is large.

### A3. Bagging from scratch

In [ ]:
class BaggingFromScratch:
    """Bagging classifier using bootstrap resampling of the training set."""

    def __init__(self, n_estimators=50, max_depth=None, random_state=0):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.rng          = np.random.default_rng(random_state)
        self.trees_       = []

    def fit(self, X, y):
        n = X.shape[0]
        for _ in range(self.n_estimators):
            idx  = self.rng.integers(0, n, size=n)      # bootstrap sample
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)
        return self

    def predict(self, X):
        # Majority vote across all trees
        votes = np.array([t.predict(X) for t in self.trees_])  # (B, n_test)
        from scipy.stats import mode
        result = mode(votes, axis=0)
        return result.mode.ravel()

from scipy.stats import mode   # pre-import to avoid issues inside class

bag = BaggingFromScratch(n_estimators=100, max_depth=None, random_state=1)
bag.fit(X_tr, y_tr)
bag_acc = accuracy_score(y_te, bag.predict(X_te))

single_tree = DecisionTreeClassifier(max_depth=None, random_state=1)
single_tree.fit(X_tr, y_tr)
tree_acc = accuracy_score(y_te, single_tree.predict(X_te))

print(f"Single unpruned tree accuracy: {tree_acc:.4f}")
print(f"Bagging (100 trees) accuracy:  {bag_acc:.4f}")
print(f"Improvement: {bag_acc - tree_acc:+.4f}")

In [ ]:
# Accuracy as a function of B
B_range = [1, 5, 10, 20, 50, 100, 150, 200]
acc_vs_B = []
for B in B_range:
    b = BaggingFromScratch(n_estimators=B, random_state=2)
    b.fit(X_tr, y_tr)
    acc_vs_B.append(accuracy_score(y_te, b.predict(X_te)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(B_range, acc_vs_B, marker='s', color='steelblue')
ax.axhline(tree_acc, color='tomato', ls='--', label='Single tree')
ax.set_xlabel('B (number of trees)')
ax.set_ylabel('Test accuracy')
ax.set_title('Bagging accuracy vs number of trees (Digits)')
ax.legend()
plt.tight_layout()
plt.show()


## Part B — Random Forests: OOB error and tuning $m$

A Random Forest adds feature subsampling to bagging.  At each split, only $m$
features are considered (default $m = \lfloor\sqrt{p}\rfloor$ for
classification).  This reduces $\rho$ and pushes the variance floor down.

Out-of-bag (OOB) error provides a free internal validation estimate: each sample
is predicted by the trees that did not bootstrap-include it.

### B1. OOB error as B grows

In [ ]:
# oob_score=True records the OOB error during training
rf_oob = RandomForestClassifier(
    n_estimators=200, oob_score=True, warm_start=True, random_state=3
)

oob_errors = []
B_range_rf = list(range(10, 201, 10))
for B in B_range_rf:
    rf_oob.set_params(n_estimators=B)
    rf_oob.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_oob.oob_score_)

rf_final = RandomForestClassifier(n_estimators=200, random_state=3)
rf_final.fit(X_tr, y_tr)
test_err = 1 - accuracy_score(y_te, rf_final.predict(X_te))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(B_range_rf, oob_errors, label='OOB error', color='steelblue')
ax.axhline(test_err, ls='--', color='tomato', label=f'Test error (B=200)')
ax.set_xlabel('B (number of trees)')
ax.set_ylabel('Error rate')
ax.set_title('OOB vs test error as B grows (Digits)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Test error (B=200):      {test_err:.4f}")

### B2. Tuning the feature subsample size $m$

In [ ]:
p = X_tr.shape[1]                          # 64 features
m_values = [1, 2, 4, 8, 16, 32, 64]       # sqrt(64)=8 is the default

oob_by_m  = []
test_by_m = []

for m in m_values:
    rf_m = RandomForestClassifier(
        n_estimators=150, max_features=m, oob_score=True, random_state=4
    )
    rf_m.fit(X_tr, y_tr)
    oob_by_m.append(1 - rf_m.oob_score_)
    test_by_m.append(1 - accuracy_score(y_te, rf_m.predict(X_te)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(m_values, oob_by_m,  marker='o', label='OOB error',  color='steelblue')
ax.plot(m_values, test_by_m, marker='s', label='Test error', color='tomato')
ax.axvline(int(np.sqrt(p)), ls=':', color='grey', label=f'sqrt(p) = {int(np.sqrt(p))}')
ax.set_xlabel('m (max_features per split)')
ax.set_ylabel('Error rate')
ax.set_title('RF error vs m (Digits, B=150)')
ax.legend()
plt.tight_layout()
plt.show()

best_m = m_values[np.argmin(oob_by_m)]
print(f"Best m by OOB: {best_m}  (default sqrt(p) = {int(np.sqrt(p))})")

**Exercise B.** The OOB error curve as a function of $m$ typically shows a
U-shape.  Explain qualitatively why both very small $m$ and $m = p$ (full
bagging) lead to higher error.

## Part C — Feature importance: MDI vs permutation importance

The lecture described two measures of variable importance:

- **MDI** (mean decrease in impurity): average reduction in Gini impurity
  over all splits on that feature, weighted by the number of samples reaching
  each node.  Fast, but biased towards high-cardinality continuous features.
- **Permutation importance**: decrease in test accuracy when a single feature's
  values are randomly shuffled.  Slower but unbiased.

With 64 pixel features on an 8×8 grid we can visualise both as heatmaps.

In [ ]:
rf_imp = RandomForestClassifier(n_estimators=200, random_state=5)
rf_imp.fit(X_tr, y_tr)

# MDI importances
mdi = rf_imp.feature_importances_

# Permutation importances on test set
perm = permutation_importance(rf_imp, X_te, y_te, n_repeats=10, random_state=5)
perm_mean = perm.importances_mean

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, imp, title in zip(
    axes,
    [mdi, perm_mean],
    ['MDI importance', 'Permutation importance (test)']
):
    im = ax.imshow(imp.reshape(8, 8), cmap='hot')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Feature importances — Digits (8x8 pixel grid)', y=1.01)
plt.tight_layout()
plt.show()
print(f"Top-5 pixels by MDI:         {np.argsort(mdi)[-5:][::-1]}")
print(f"Top-5 pixels by permutation: {np.argsort(perm_mean)[-5:][::-1]}")

**Exercise C.** Look at the two heatmaps.  Corner pixels tend to appear less
important than central pixels.  Why does this make intuitive sense for digit
recognition?  Do MDI and permutation importance agree on which pixels matter
most?

## Part D — Gradient boosting with early stopping

Gradient boosting builds trees sequentially, each fitting the residuals of the
current ensemble.  Unlike bagging, the trees are shallow (low variance) and the
method reduces bias rather than variance.  Early stopping monitors a validation
loss to avoid overfitting to the training residuals.

### D1. Learning rate and number of stages

In [ ]:
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=6, stratify=y_tr
)

learning_rates = [0.5, 0.1, 0.05, 0.01]
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(7, 4))

for lr, col in zip(learning_rates, colors):
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=lr,
        max_depth=3, random_state=7
    )
    gb.fit(X_tr2, y_tr2)
    val_errors = [
        1 - accuracy_score(y_val, y_pred)
        for y_pred in gb.staged_predict(X_val)
    ]
    ax.plot(val_errors, label=f'lr={lr}', color=col, lw=1.2)

ax.set_xlabel('Boosting stage')
ax.set_ylabel('Validation error')
ax.set_title('GB: validation error by learning rate (Digits)')
ax.legend()
plt.tight_layout()
plt.show()


### D2. Early stopping

In [ ]:
# Manual early stopping: record the stage with minimum validation error
gb_es = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.1,
    max_depth=3, random_state=8
)
gb_es.fit(X_tr2, y_tr2)

val_errors_es = [
    1 - accuracy_score(y_val, y_pred)
    for y_pred in gb_es.staged_predict(X_val)
]
best_stage = int(np.argmin(val_errors_es)) + 1
best_val_err = val_errors_es[best_stage - 1]

# Refit with optimal number of stages on the full training set
gb_best = GradientBoostingClassifier(
    n_estimators=best_stage, learning_rate=0.1,
    max_depth=3, random_state=8
)
gb_best.fit(X_tr, y_tr)
test_acc_gb = accuracy_score(y_te, gb_best.predict(X_te))

print(f"Optimal stage (by val error):  {best_stage}")
print(f"Validation error at optimum:   {best_val_err:.4f}")
print(f"Test accuracy (refitted):      {test_acc_gb:.4f}")

**Exercise D.** Suppose you increase `learning_rate` from 0.1 to 0.5 while
keeping `n_estimators` fixed.  What do you expect to happen to the training
error, and why might the test error increase?

## Part E — Side-by-side comparison

We now train all three ensemble families with comparable budgets and compare
their test accuracy and training time on the Digits dataset.

In [ ]:
import time

models = {
    'Bagging (B=200)': BaggingClassifier(
        n_estimators=200, random_state=9
    ),
    'Random Forest (B=200)': RandomForestClassifier(
        n_estimators=200, random_state=9
    ),
    'Gradient Boosting (200 stages)': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=9
    ),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    acc = accuracy_score(y_te, model.predict(X_te))
    results[name] = {'accuracy': acc, 'time_s': elapsed}
    print(f"{name:<40} acc={acc:.4f}  time={elapsed:.2f}s")

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
times = [results[n]['time_s']   for n in names]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].barh(names, accs, color=['steelblue', 'seagreen', 'tomato'])
axes[0].set_xlabel('Test accuracy')
axes[0].set_title('Accuracy comparison')
axes[0].set_xlim(0.9, 1.0)

axes[1].barh(names, times, color=['steelblue', 'seagreen', 'tomato'])
axes[1].set_xlabel('Training time (s)')
axes[1].set_title('Training time comparison')

plt.tight_layout()
plt.show()


**Exercise E.** Gradient boosting is typically slower to train than a Random
Forest of the same size.  Explain why, in terms of the sequential vs parallel
structure of the two methods.

---

## Take-home exercises

**Exercise 1 — Correlation and the variance floor.**
Set $\sigma^2 = 0.25$ and $B = 200$.  Using `simulate_ensemble_variance` from
Part A, compute the empirical variance for $\rho \in \{0, 0.1, 0.2, \ldots, 0.9\}$
and plot it against the theoretical formula $\rho\,\sigma^2 + (1-\rho)\sigma^2/B$.
Comment on how well the formula matches the simulation.

**Exercise 2 — OOB as a tuning tool.**
Using a grid of `max_depth` values $\{3, 5, 10, 15, \text{None}\}$ and
`max_features` values $\{4, 8, 16, 32\}$, train a Random Forest ($B = 150$,
`oob_score=True`) for each combination and record OOB error.  Plot a heatmap of
OOB errors and identify the best pair.  Confirm on the test set.

**Exercise 3 — Permutation importance with correlated features.**
Add 10 copies of pixel 36 (the centre pixel) to the Digits dataset as synthetic
duplicates, then recompute MDI and permutation importance.  Explain the
difference in how each measure responds to the added redundant features.

**Exercise 4 — Gradient boosting tree depth.**
Fix `n_estimators=200`, `learning_rate=0.1` and vary `max_depth` over
$\{1, 2, 3, 5, 8\}$.  For each depth, record both training accuracy and test
accuracy.  At what depth does overfitting become apparent?  Relate this to the
bias-variance trade-off described in the lecture.